# PAP_NER + PhoBERT-v2 + CRF — Kaggle 1-epoch verification

Notebook này là bản **Kaggle-ready** để kiểm tra toàn bộ pipeline với:

- Model: **`vinai/phobert-base-v2`**
- Dataset: **`Leekien0108/PAP_NER`**, giữ nguyên revision của notebook gốc để so sánh công bằng.
- Training: **1 epoch** để verify setup/pipeline trước khi chạy benchmark dài.
- Effective batch size: **32** bằng gradient accumulation.
- Precision: **FP16** trên GPU.
- Max sequence length: **256**.
- PhoBERT LR: **2e-5**.
- Linear + CRF head LR: **1e-3**.
- AdamW + linear warmup/decay, warmup ratio **0.1**.
- Weight decay **0.01**, gradient clipping **1.0**, dropout **0.1**.
- Seed: **42**.
- Best checkpoint: chọn theo **dev Strict Micro F1**.

## Kaggle Settings bắt buộc

1. Mở notebook trên Kaggle.
2. **Settings → Accelerator → GPU** (T4/P100 hoặc GPU khác). Nếu vừa đổi từ CPU sang GPU, hãy **Restart Session** trước khi Run All.
3. **Internet → ON** để tải PhoBERT-v2 và PAP_NER từ Hugging Face.
4. Chạy **Run All** từ trên xuống.
5. Cell smoke test sẽ kiểm tra VRAM trước khi train thật.
6. Kết quả và file ZIP được lưu trong **`/kaggle/working`**.

> **Bootstrap PyTorch:** nếu Kaggle image đã có PyTorch thì notebook giữ nguyên. Nếu thiếu `torch`, notebook tự cài **PyTorch 2.7.0 CUDA 11.8** từ wheel chính thức rồi mới kiểm tra GPU. Internet phải bật để fallback này hoạt động.


> **GPU detection:** bản FIX dùng `torch.cuda.is_available()` / `torch.cuda.get_device_properties()`; không phụ thuộc executable `nvidia-smi`.


## 1. GPU và cấu hình thí nghiệm


In [1]:
from pathlib import Path
import json, os, shutil, subprocess, sys, time

# =========================
# USER CONFIG — KAGGLE TEST
# =========================
TRIAL_EPOCHS = 1
NOTEBOOK_BUILD = "FINAL_NO_PLOT_V4"
RESUME = False
MODEL_NAME = "vinai/phobert-base-v2"

HF_DATASET_REPO = "Leekien0108/PAP_NER"
HF_DATASET_REVISION = "7a0f6b1233667702925baa69c07c55d5d26eb6cd"
CUSTOM_DATA_DIR = None  # optional: folder containing train_data.txt/dev_data.txt/test_data.txt

# =========================
# Kaggle / Linux / GPU detection
# =========================
# Do NOT depend on the external `nvidia-smi` executable. Kaggle images can
# change, while PyTorch is the API the training loop actually uses.
if sys.platform != "linux":
    raise RuntimeError("Notebook này được cấu hình cho Kaggle/Linux.")

# Bootstrap PyTorch only when the Kaggle image does not provide it.
# CUDA 11.8 wheel is intentionally used for broad compatibility with Kaggle T4/P100.
def _ensure_torch():
    try:
        import torch as _torch
        print(f"PyTorch preinstalled: {_torch.__version__} (CUDA runtime: {_torch.version.cuda})")
        return _torch
    except ModuleNotFoundError:
        print("PyTorch is missing. Installing official torch==2.7.0 CUDA 11.8 wheel...")
        cmd = [
            sys.executable, "-m", "pip", "install",
            "--disable-pip-version-check", "--no-cache-dir",
            "torch==2.7.0",
            "--index-url", "https://download.pytorch.org/whl/cu118",
        ]
        try:
            subprocess.run(cmd, check=True)
        except subprocess.CalledProcessError as exc:
            raise RuntimeError(
                "Không cài được PyTorch. Trên Kaggle hãy bật Internet rồi Run All lại. "
                "Nếu notebook thuộc competition bắt buộc Internet OFF, cần attach wheel torch làm Dataset/Input."
            ) from exc
        import importlib
        importlib.invalidate_caches()
        try:
            import torch as _torch
        except Exception as exc:
            raise RuntimeError(
                "Đã chạy pip install torch nhưng vẫn không import được trong kernel hiện tại."
            ) from exc
        print(f"PyTorch installed: {_torch.__version__} (CUDA runtime: {_torch.version.cuda})")
        return _torch
    except Exception as exc:
        raise RuntimeError(
            "Có PyTorch nhưng import bị lỗi (không phải ModuleNotFoundError). "
            "Hãy Restart Session để tránh môi trường package bị trộn."
        ) from exc

torch = _ensure_torch()

if not torch.cuda.is_available():
    cuda_visible = os.environ.get("CUDA_VISIBLE_DEVICES", "<unset>")
    raise RuntimeError(
        "PyTorch không thấy CUDA/GPU. Đây không phải lỗi PhoBERT/CRF. "
        "Trên Kaggle hãy mở Settings → Accelerator → chọn GPU, rồi Restart Session/Run All. "
        f"torch={torch.__version__}, torch.version.cuda={torch.version.cuda}, "
        f"CUDA_VISIBLE_DEVICES={cuda_visible}"
    )

GPU_INDEX = 0
GPU_NAME = torch.cuda.get_device_name(GPU_INDEX)
GPU_VRAM_GIB = torch.cuda.get_device_properties(GPU_INDEX).total_memory / (1024 ** 3)
GPU_CAPABILITY = torch.cuda.get_device_capability(GPU_INDEX)

# Conservative profiles. Always keep effective train batch = 32.
if GPU_VRAM_GIB >= 22:
    TRAIN_BATCH_SIZE = 16
    EVAL_BATCH_SIZE = 32
    GRAD_ACCUM_STEPS = 2
    GRADIENT_CHECKPOINTING = False
elif GPU_VRAM_GIB >= 14:
    # Kaggle T4/P100 commonly lands here.
    TRAIN_BATCH_SIZE = 8
    EVAL_BATCH_SIZE = 16
    GRAD_ACCUM_STEPS = 4
    GRADIENT_CHECKPOINTING = False
elif GPU_VRAM_GIB >= 10:
    TRAIN_BATCH_SIZE = 4
    EVAL_BATCH_SIZE = 8
    GRAD_ACCUM_STEPS = 8
    GRADIENT_CHECKPOINTING = True
else:
    TRAIN_BATCH_SIZE = 2
    EVAL_BATCH_SIZE = 4
    GRAD_ACCUM_STEPS = 16
    GRADIENT_CHECKPOINTING = True

# =========================
# Paths
# =========================
KAGGLE_WORKING = Path("/kaggle/working")
NOTEBOOK_DIR = KAGGLE_WORKING if KAGGLE_WORKING.exists() else Path.cwd()
default_root = NOTEBOOK_DIR / "ner-pap"
WORK_ROOT = Path(os.environ.get("NER_WORK_ROOT", str(default_root))).resolve()
WORK_ROOT.mkdir(parents=True, exist_ok=True)

RUN_NAME = "papner_phobert_base_v2_crf_1ep_seed42_kaggle"
RUN_DIR = WORK_ROOT / "runs" / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)

# =========================
# Experiment config
# =========================
CFG = {
    "model_name_or_path": MODEL_NAME,
    "max_seq_length": 256,
    "train_batch_size": TRAIN_BATCH_SIZE,
    "eval_batch_size": EVAL_BATCH_SIZE,
    "gradient_accumulation_steps": GRAD_ACCUM_STEPS,
    "gradient_checkpointing": GRADIENT_CHECKPOINTING,
    "precision": "fp16",
    "learning_rate": 2e-5,
    "classifier_learning_rate": 1e-3,
    "epochs": TRIAL_EPOCHS,
    "early_stop": 3,
    "warmup_proportion": 0.1,
    "weight_decay": 0.01,
    "adam_epsilon": 1e-6,
    "max_grad_norm": 1.0,
    "dropout": 0.1,
    "seed": 42,
    "num_workers": 2,
    "device": "cuda",
    "smoke_batches": 2,
    "train_limit": None,
    "dev_limit": None,
    "cpu_threads": 4,
    "selection_metric": "entity_strict_micro_f1",
    "resume": RESUME,
}

assert CFG["train_batch_size"] * CFG["gradient_accumulation_steps"] == 32
assert CFG["precision"] == "fp16"
assert CFG["epochs"] == 1
assert CFG["max_seq_length"] == 256
assert CFG["model_name_or_path"] == "vinai/phobert-base-v2"

CFG.update(
    dataset_repo=HF_DATASET_REPO,
    dataset_requested_revision=HF_DATASET_REVISION,
    dataset_revision=None,
    data_files={},
    output_dir=str(RUN_DIR),
    cache_dir=str(WORK_ROOT / "feature-cache"),
)

CONFIG_PATH = RUN_DIR / "launch_config.json"
CONFIG_PATH.write_text(json.dumps(CFG, ensure_ascii=False, indent=2), encoding="utf-8")

# Use Kaggle's current Python + PyTorch/CUDA. Do NOT reinstall torch.
ENV_PYTHON = Path(sys.executable)
RUNNER = WORK_ROOT / "papner_runner.py"

ENV = dict(
    os.environ,
    HF_HOME=str(WORK_ROOT / "hf-cache"),
    TOKENIZERS_PARALLELISM="false",
    PYTHONUNBUFFERED="1",
    PYTHONIOENCODING="utf-8",
    CUDA_VISIBLE_DEVICES=os.environ.get("CUDA_VISIBLE_DEVICES", "0"),
    MPLBACKEND="Agg",
    PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True",
)

print(f"GPU: {GPU_NAME}")
print(f"VRAM: {GPU_VRAM_GIB:.1f} GiB")
print(f"CUDA capability: {GPU_CAPABILITY}")
print(f"Torch: {torch.__version__} | CUDA runtime: {torch.version.cuda}")
print(f"Notebook build: {NOTEBOOK_BUILD}")
print(f"Python: {ENV_PYTHON}")
print(f"Model: {MODEL_NAME}")
print(f"Train batch: {TRAIN_BATCH_SIZE}")
print(f"Eval batch: {EVAL_BATCH_SIZE}")
print(f"Gradient accumulation: {GRAD_ACCUM_STEPS}")
print(f"Effective train batch: {TRAIN_BATCH_SIZE * GRAD_ACCUM_STEPS}")
print(f"Gradient checkpointing: {GRADIENT_CHECKPOINTING}")
print(f"Epochs: {TRIAL_EPOCHS}")
print(f"Run dir: {RUN_DIR}")


PyTorch is missing. Installing official torch==2.7.0 CUDA 11.8 wheel...


Looking in indexes: https://download.pytorch.org/whl/cu118


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/23.2 MB ? eta -:--:--

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 251.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 875.6/875.6 kB 627.0 MB/s eta 0:00:00


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 600.0 MB/s eta 0:00:00


...

   ━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━━━━━━ 96.7/156.4 MB 160.0 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━━ 122.4/156.4 MB 151.7 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.4/156.4 MB 145.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 110.3 MB/s eta 0:00:00


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 800.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.2/536.2 kB 584.2 MB/s eta 0:00:00


PyTorch installed: 2.7.0+cu118 (CUDA runtime: 11.8)
GPU: Tesla T4
VRAM: 14.6 GiB
CUDA capability: (7, 5)
Torch: 2.7.0+cu118 | CUDA runtime: 11.8
Notebook build: FINAL_NO_PLOT_V4
Python: /usr/bin/python
Model: vinai/phobert-base-v2
Train batch: 8
Eval batch: 16
Gradient accumulation: 4
Effective train batch: 32
Gradient checkpointing: False
Epochs: 1
Run dir: /kaggle/working/ner-pap/runs/papner_phobert_base_v2_crf_1ep_seed42_kaggle


## 2. Chuẩn bị môi trường Kaggle

Notebook **không tạo venv**. Cell cấu hình phía trên ưu tiên dùng PyTorch có sẵn; chỉ khi `import torch` báo thiếu module thì mới cài `torch==2.7.0` từ official CUDA 11.8 wheel. Sau khi GPU đã được xác nhận, cell này cài các dependency còn lại của pipeline.

Cell bên dưới chỉ cài/ghim các dependency trực tiếp của pipeline:

- `transformers==4.44.2`
- `huggingface_hub==0.36.2`
- `pytorch-crf==0.7.2`
- `seqeval==1.2.2`
- `nervaluate==0.2.0`
- `tensorboard`, `tabulate`

Sau đó chạy CUDA FP16 smoke operation để xác nhận GPU hoạt động.


In [2]:
import platform

print("Platform:", platform.platform())
print("Python:", sys.version)
print("Using Kaggle/system Python:", ENV_PYTHON)

# Torch was verified/bootstraped above. Install only the remaining Python dependencies.
packages = [
    "transformers==4.44.2",
    "huggingface_hub==0.36.2",
    "pytorch-crf==0.7.2",
    "seqeval==1.2.2",
    "nervaluate==0.2.0",
    "tensorboard>=2.15,<3",
    "tabulate==0.9.0",
    # Required by advanced evaluation tables.
    "pandas>=2.1,<3",
]

subprocess.run(
    [str(ENV_PYTHON), "-m", "pip", "install", "-q", "--disable-pip-version-check", *packages],
    check=True,
    env=ENV,
)

gpu_check = r"""
import torch
import transformers
import huggingface_hub
import torchcrf
import seqeval
import pandas

print("torch:", torch.__version__)
print("torch path:", torch.__file__)
print("CUDA runtime:", torch.version.cuda)
print("transformers:", transformers.__version__)
print("huggingface_hub:", huggingface_hub.__version__)
print("pandas:", pandas.__version__)
assert torch.cuda.is_available(), "CUDA unavailable — enable GPU accelerator in Kaggle"
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GiB:", torch.cuda.get_device_properties(0).total_memory / 2**30)
print("Capability:", torch.cuda.get_device_capability(0))

x = torch.randn((512, 512), device="cuda", dtype=torch.float16)
y = x @ x
assert torch.isfinite(y).all(), "FP16 CUDA smoke operation failed"
print("FP16 CUDA smoke op: OK")
"""
subprocess.run([str(ENV_PYTHON), "-c", gpu_check], check=True, env=ENV)


Platform: Linux-6.18.48+-x86_64-with-glibc2.35
Python: 3.10.12 (main, Nov  6 2024, 20:22:13) [GCC 11.4.0]
Using Kaggle/system Python: /usr/bin/python


torch: 2.7.0+cu118
torch path: /usr/local/lib/python3.10/dist-packages/torch/__init__.py
CUDA runtime: 11.8
transformers: 4.44.2
huggingface_hub: 0.36.2
pandas: 2.3.3
GPU: Tesla T4
VRAM GiB: 14.56219482421875
Capability: (7, 5)


FP16 CUDA smoke op: OK


CompletedProcess(args=['/usr/bin/python', '-c', '\nimport torch\nimport transformers\nimport huggingface_hub\nimport torchcrf\nimport seqeval\nimport pandas\n\nprint("torch:", torch.__version__)\nprint("torch path:", torch.__file__)\nprint("CUDA runtime:", torch.version.cuda)\nprint("transformers:", transformers.__version__)\nprint("huggingface_hub:", huggingface_hub.__version__)\nprint("pandas:", pandas.__version__)\nassert torch.cuda.is_available(), "CUDA unavailable — enable GPU accelerator in Kaggle"\nprint("GPU:", torch.cuda.get_device_name(0))\nprint("VRAM GiB:", torch.cuda.get_device_properties(0).total_memory / 2**30)\nprint("Capability:", torch.cuda.get_device_capability(0))\n\nx = torch.randn((512, 512), device="cuda", dtype=torch.float16)\ny = x @ x\nassert torch.isfinite(y).all(), "FP16 CUDA smoke operation failed"\nprint("FP16 CUDA smoke op: OK")\n'], returncode=0)

## 3. Tải PAP_NER từ Hugging Face

Notebook dùng đúng ba split `train / dev / test` đã phát hành. Dataset được ghim theo commit SHA để lần chạy có thể tái lập.

Nếu muốn dùng file local, đặt `CUSTOM_DATA_DIR` ở cell cấu hình thành thư mục chứa:

- `train_data.txt`
- `dev_data.txt`
- `test_data.txt`


In [3]:
if CUSTOM_DATA_DIR is None:
    fetch_script = r"""
import json, sys
from pathlib import Path
from huggingface_hub import HfApi, snapshot_download

repo_id, revision, output_root = sys.argv[1:]
info = HfApi().dataset_info(repo_id, revision=revision)
sha = info.sha
if not sha:
    raise RuntimeError('Hugging Face did not return a dataset commit SHA')
files = [item.rfilename for item in info.siblings]
choices = {
    'train': {'train_data.txt', 'train.txt'},
    'dev': {'dev_data.txt', 'dev.txt', 'validation_data.txt', 'validation.txt'},
    'test': {'test_data.txt', 'test.txt'},
}
selected = {}
for split, names in choices.items():
    matches = [name for name in files if Path(name).name in names]
    if len(matches) != 1:
        raise RuntimeError(f'Expected exactly one CoNLL file for {split}; found {matches}. '
                           f'Dataset files: {files}')
    selected[split] = matches[0]
root = Path(snapshot_download(repo_id=repo_id, repo_type='dataset', revision=sha,
                              local_dir=str(Path(output_root) / sha),
                              allow_patterns=list(selected.values())))
paths = {split: str((root / name).resolve()) for split, name in selected.items()}
for split, path in paths.items():
    if not Path(path).is_file() or Path(path).stat().st_size == 0:
        raise RuntimeError(f'Missing or empty {split} file: {path}')
print(json.dumps({'sha': sha, 'files': paths}, ensure_ascii=False))
"""
    try:
        fetched = subprocess.check_output(
            [str(ENV_PYTHON), '-c', fetch_script, HF_DATASET_REPO, HF_DATASET_REVISION,
             str(WORK_ROOT / 'hf-datasets')], text=True, env=ENV)
    except subprocess.CalledProcessError as exc:
        raise RuntimeError('Không tải được dataset từ Hugging Face. Kiểm tra kết nối và '
                           'HF_TOKEN trong môi trường Jupyter nếu repo riêng tư.') from exc
    manifest = json.loads(fetched)
    CFG['dataset_revision'] = manifest['sha']
    CFG['data_files'] = manifest['files']
else:
    data_dir = Path(CUSTOM_DATA_DIR).resolve()
    CFG['dataset_repo'] = 'custom-local-files'
    CFG['dataset_revision'] = 'custom-local-files'
    CFG['data_files'] = {split: str(data_dir / f'{split}_data.txt')
                         for split in ('train', 'dev', 'test')}
CONFIG_PATH.write_text(json.dumps(CFG, ensure_ascii=False, indent=2), encoding='utf-8')
print('Dataset:', HF_DATASET_REPO, 'revision:', CFG['dataset_revision'])
for split in ('train', 'dev', 'test'):
    path = Path(CFG['data_files'][split])
    assert path.is_file() and path.stat().st_size > 0, f'Thiếu dữ liệu: {path}'
    print(f'{split}: {path} ({path.stat().st_size / 2**20:.2f} MiB)')
with Path(CFG['data_files']['dev']).open(encoding='utf-8') as stream:
    for _ in range(6):
        print(stream.readline().rstrip())

# Verify released split sizes from the reproduction setup.
EXPECTED_SENTENCES = {"train": 142_218, "dev": 10_305, "test": 10_278}

def count_conll_sentences(path):
    count = 0
    inside = False
    with Path(path).open(encoding="utf-8-sig") as stream:
        for line in stream:
            if line.strip():
                inside = True
            elif inside:
                count += 1
                inside = False
    if inside:
        count += 1
    return count

for split, expected in EXPECTED_SENTENCES.items():
    actual = count_conll_sentences(CFG["data_files"][split])
    if actual != expected:
        raise RuntimeError(
            f"Split {split} có {actual:,} câu, expected {expected:,}. "
            "Dataset hiện tại không khớp reproduction setup."
        )
    print(f"{split}: {actual:,} sentences ✓")


Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Fetching 3 files:  33%|███▎      | 1/3 [00:00<00:00,  3.44it/s]

Fetching 3 files: 100%|██████████| 3/3 [00:01<00:00,  1.87it/s]


Dataset: Leekien0108/PAP_NER revision: 7a0f6b1233667702925baa69c07c55d5d26eb6cd
train: /kaggle/working/ner-pap/hf-datasets/7a0f6b1233667702925baa69c07c55d5d26eb6cd/train_data.txt (61.51 MiB)
dev: /kaggle/working/ner-pap/hf-datasets/7a0f6b1233667702925baa69c07c55d5d26eb6cd/dev_data.txt (4.83 MiB)
test: /kaggle/working/ner-pap/hf-datasets/7a0f6b1233667702925baa69c07c55d5d26eb6cd/test_data.txt (4.80 MiB)
Đăng_ký N B-NP O O
thành_lập V B-VP O O
doanh_nghiệp V I-VP B-ĐT O
tư_nhân A O I-ĐT O
do E B-PP O O
cơ_quan T O O O


train: 142,218 sentences ✓
dev: 10,305 sentences ✓
test: 10,278 sentences ✓


## 4. Runner — phần A: dữ liệu và căn nhãn subword

Runner đọc PAP_NER, dùng PhoBERT BPE và chỉ đưa **subword đầu tiên của mỗi từ** vào CRF. Các subword còn lại bị bỏ qua trong loss/evaluation, tương đương vai trò `X/ignore` trong setup reproduction.


In [4]:
RUNNER_PARTS = []
RUNNER_PARTS.append(r'''
"""Standalone PhoBERT-CRF runner for PAP_NER reproduction on Linux GPUs."""
import argparse
import hashlib
import itertools
import json
import math
import os
import random
import sys
import time
import traceback
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset, Subset
from torch.utils.tensorboard import SummaryWriter
from torchcrf import CRF
from transformers import AutoConfig, AutoTokenizer, RobertaForTokenClassification
from transformers import get_linear_schedule_with_warmup
from sklearn.metrics import accuracy_score, classification_report, f1_score
from seqeval.metrics import classification_report as entity_report
from seqeval.metrics import f1_score as entity_f1
from seqeval.metrics import precision_score as entity_precision
from seqeval.metrics import recall_score as entity_recall
from seqeval.scheme import IOB2
from tqdm.auto import tqdm

LABELS = ['O', 'B-CQ', 'I-CQ', 'B-ĐT', 'I-ĐT', 'B-VBPL', 'I-VBPL',
          'B-NG', 'I-NG', 'B-SL', 'I-SL']


def write_json(path, value):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding='utf-8')
    os.replace(tmp, path)


def read_conll(path):
    """PAP_NER NER is column 4; column 5 is NOT the target."""
    words, tags = [], []
    with Path(path).open(encoding='utf-8-sig') as stream:
        for line_no, line in enumerate(stream, 1):
            fields = line.split()
            if not fields:
                if words:
                    yield words, tags
                    words, tags = [], []
                continue
            if len(fields) >= 4:
                word, tag = fields[0], fields[3]
            elif len(fields) == 2:
                word, tag = fields
            else:
                raise ValueError(f'{path}:{line_no}: expected 2 or >=4 columns')
            if tag not in LABELS:
                raise ValueError(f'{path}:{line_no}: unknown label {tag!r}')
            words.append(word)
            tags.append(LABELS.index(tag))
    if words:
        yield words, tags


def encode_words(words, tags, tokenizer, max_length):
    """Keep only complete words fitting the BPE budget; align first subword."""
    ids, positions, kept_tags = [], [], []
    for word, tag in zip(words, tags):
        pieces = tokenizer.encode(word, add_special_tokens=False)
        if not pieces:
            pieces = [tokenizer.unk_token_id]
        if len(ids) + len(pieces) > max_length - 2:
            break
        positions.append(len(ids) + 1)  # +1 for <s>
        ids.extend(pieces)
        kept_tags.append(tag)
    if not positions:
        raise ValueError(f'First word exceeds BPE budget: {words[:1]}')
    return {'input_ids': [tokenizer.bos_token_id] + ids + [tokenizer.eos_token_id],
            'valid_ids': positions, 'labels': kept_tags}


class Features(Dataset):
    def __init__(self, examples):
        self.examples = examples

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return self.examples[idx]


class Collator:
    def __init__(self, pad_id):
        self.pad_id = pad_id

    def __call__(self, examples):
        # Dynamic padding on CPU; GPU transfer happens in the parent process.
        length = math.ceil(max(len(x['input_ids']) for x in examples) / 8) * 8
        words = max(len(x['valid_ids']) for x in examples)
        result = {'input_ids': torch.full((len(examples), length), self.pad_id, dtype=torch.long),
                  'attention_mask': torch.zeros(len(examples), length, dtype=torch.long),
                  'valid_ids': torch.zeros(len(examples), words, dtype=torch.long),
                  'labels': torch.zeros(len(examples), words, dtype=torch.long),
                  'label_masks': torch.zeros(len(examples), words, dtype=torch.bool)}
        for i, ex in enumerate(examples):
            n, w = len(ex['input_ids']), len(ex['valid_ids'])
            result['input_ids'][i, :n] = torch.tensor(ex['input_ids'])
            result['attention_mask'][i, :n] = 1
            result['valid_ids'][i, :w] = torch.tensor(ex['valid_ids'])
            result['labels'][i, :w] = torch.tensor(ex['labels'])
            result['label_masks'][i, :w] = True
        return result
''')

## 5. Runner — phần B: PhoBERT + CRF, DataLoader và metrics

CRF chạy ở FP32 để ổn định log-sum-exp; encoder/linear chạy dưới autocast FP16.

Evaluation trả về:

- Strict Micro Precision
- Strict Micro Recall
- Strict Micro F1
- Strict Macro F1
- BIO token Macro F1


In [5]:
RUNNER_PARTS.append(r'''
class PhoBertCrf(RobertaForTokenClassification):
    # Preserve baseline state_dict names: roberta.*, classifier.*, crf.*.
    def __init__(self, config):
        super().__init__(config)
        self.crf = CRF(config.num_labels, batch_first=True)

    def forward(self, input_ids, attention_mask, valid_ids, label_masks, labels=None,
                decode=False):
        hidden = self.roberta(input_ids=input_ids, attention_mask=attention_mask)[0]
        rows = torch.arange(hidden.shape[0], device=hidden.device).unsqueeze(1)
        emissions = self.classifier(self.dropout(hidden[rows, valid_ids]))
        # Encoder/linear use AMP; logsumexp and CRF transitions use float32.
        with torch.autocast(device_type=hidden.device.type, enabled=False):
            emissions = emissions.float()
            loss = -self.crf(emissions, labels, mask=label_masks, reduction='sum') if labels is not None else None
            tags = self.crf.decode(emissions, mask=label_masks) if decode else None
        return SimpleNamespace(loss=loss, tags=tags)


def feature_dataset(cfg, split, tokenizer):
    source = Path(cfg['data_files'][split])
    fingerprint = hashlib.sha256(source.read_bytes()).hexdigest()
    limit = cfg.get(f'{split}_limit')
    key = hashlib.sha256(f'word-prefix-v2:{fingerprint}:{cfg["model_name_or_path"]}:{cfg["max_seq_length"]}:{limit}'.encode()).hexdigest()[:20]
    cache = Path(cfg['cache_dir']) / f'{split}-{key}.pt'
    cache.parent.mkdir(parents=True, exist_ok=True)
    if cache.exists():
        packed = torch.load(cache, map_location='cpu', weights_only=True)
    else:
        examples, truncated, discarded = [], 0, 0
        sentences = read_conll(source)
        if limit is not None:
            sentences = itertools.islice(sentences, limit)
        for words, tags in tqdm(sentences, desc=f'Tokenize {split}'):
            ex = encode_words(words, tags, tokenizer, cfg['max_seq_length'])
            truncated += int(len(ex['labels']) < len(tags))
            discarded += len(tags) - len(ex['labels'])
            examples.append(ex)
        packed = {'examples': examples, 'truncated_sentences': truncated,
                  'discarded_words': discarded, 'sha256': fingerprint, 'limit': limit}
        temp = cache.with_suffix('.tmp')
        torch.save(packed, temp)
        os.replace(temp, cache)
    if not packed['examples']:
        raise ValueError(f'{source} contains no sentences')
    print(f'{split}: {len(packed["examples"]):,} sentences (limit={limit}); '
          f'{packed["truncated_sentences"]:,} truncated; {packed["discarded_words"]:,} words excluded', flush=True)
    return Features(packed['examples']), {k: v for k, v in packed.items() if k != 'examples'}


def loader(dataset, cfg, tokenizer, train=False, seed=None):
    generator = torch.Generator().manual_seed(cfg['seed'] if seed is None else seed)
    return DataLoader(dataset, batch_size=cfg['train_batch_size'] if train else cfg['eval_batch_size'],
                      shuffle=train, generator=generator, num_workers=cfg['num_workers'],
                      pin_memory=cfg['device'] == 'cuda', collate_fn=Collator(tokenizer.pad_token_id),
                      persistent_workers=cfg['num_workers'] > 0)


def gpu_batch(batch, device):
    return {k: v.to(device, non_blocking=True) for k, v in batch.items()}


def amp(cfg, device):
    dtype = torch.bfloat16 if cfg['precision'] == 'bf16' else torch.float16
    return torch.autocast(device_type=device.type, dtype=dtype, enabled=device.type == 'cuda' and cfg['precision'] != 'fp32')


@torch.inference_mode()
def evaluate(model, iterator, cfg, device):
    model.eval()
    golds, preds, total_loss = [], [], 0.0
    for batch in tqdm(iterator, desc='Evaluate'):
        batch = gpu_batch(batch, device)
        with amp(cfg, device):
            output = model(**batch, decode=True)
        total_loss += output.loss.item()
        for gold, mask, predicted in zip(batch['labels'].cpu(), batch['label_masks'].cpu(), output.tags):
            actual = gold[mask].tolist()
            assert len(actual) == len(predicted), 'CRF alignment mismatch'
            golds.append([LABELS[i] for i in actual])
            preds.append([LABELS[i] for i in predicted])
    flat_gold = [x for sent in golds for x in sent]
    flat_pred = [x for sent in preds for x in sent]
    strict_precision = entity_precision(
        golds, preds, average='micro', mode='strict', scheme=IOB2, zero_division=0)
    strict_recall = entity_recall(
        golds, preds, average='micro', mode='strict', scheme=IOB2, zero_division=0)
    strict_micro_f1 = entity_f1(
        golds, preds, average='micro', mode='strict', scheme=IOB2, zero_division=0)
    strict_macro_f1 = entity_f1(
        golds, preds, average='macro', mode='strict', scheme=IOB2, zero_division=0)
    return {'loss_per_sentence': total_loss / len(iterator.dataset),
            'bio_accuracy': accuracy_score(flat_gold, flat_pred),
            'bio_macro_f1': f1_score(flat_gold, flat_pred, average='macro', labels=LABELS, zero_division=0),
            'entity_strict_precision_micro': strict_precision,
            'entity_strict_recall_micro': strict_recall,
            'entity_strict_micro_f1': strict_micro_f1,
            'entity_strict_macro_f1': strict_macro_f1}, golds, preds
''')

## 6. Runner — phần C: optimizer, scheduler, training, early stopping

Các điểm chính:

- PhoBERT LR `2e-5`.
- Linear + CRF head LR `1e-3`.
- AdamW, weight decay `0.01`.
- Linear scheduler + warmup `10%`.
- Gradient clipping `1.0`.
- FP16 GradScaler.
- Early stopping patience `3`.
- Best checkpoint chọn theo **dev Strict Micro F1**.


In [6]:
RUNNER_PARTS.append(r'''
def save_checkpoint(path, model, cfg, **extra):
    # CPU tensors keep saved models portable and avoid a second VRAM copy.
    args = SimpleNamespace(**cfg, task='pap_ner', model_arch='crf', label2id=LABELS,
                           id2label=dict(enumerate(LABELS)))
    data = {'model': {k: v.detach().cpu() for k, v in model.state_dict().items()},
            'classes': LABELS, 'args': args, 'config': cfg, **extra}
    path = Path(path)
    tmp = path.with_suffix('.tmp')
    torch.save(data, tmp)
    os.replace(tmp, path)


def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def setup(cfg, pretrained=True):
    device = torch.device('cuda:0' if cfg['device'] == 'cuda' else 'cpu')
    if device.type == 'cuda' and not torch.cuda.is_available():
        raise RuntimeError('CUDA unavailable to PyTorch. On Kaggle enable Settings → Accelerator → GPU, restart the session, then Run All.')
    if device.type == 'cpu' and cfg['precision'] != 'fp32':
        raise ValueError('CPU mode requires fp32 precision.')
    if device.type == 'cuda' and cfg['precision'] == 'bf16' and not torch.cuda.is_bf16_supported():
        raise RuntimeError('BF16 unsupported; change precision to fp16.')
    seed_all(cfg['seed'])
    if device.type == 'cpu':
        torch.set_num_threads(cfg.get('cpu_threads', 4))
    if device.type == 'cuda':
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
    tokenizer = AutoTokenizer.from_pretrained(cfg['model_name_or_path'], use_fast=False)
    config = AutoConfig.from_pretrained(cfg['model_name_or_path'], num_labels=len(LABELS),
                                       id2label=dict(enumerate(LABELS)), label2id={v: k for k, v in enumerate(LABELS)})
    # Reproduction setup specifies dropout=0.1. Apply it explicitly instead of
    # depending on the value stored in the pretrained config.
    config.hidden_dropout_prob = cfg['dropout']
    config.attention_probs_dropout_prob = cfg['dropout']
    config.classifier_dropout = cfg['dropout']
    config._attn_implementation = 'eager'
    model = PhoBertCrf.from_pretrained(cfg['model_name_or_path'], config=config) if pretrained else PhoBertCrf(config)
    if pretrained:
        # CRF is added after the Transformers base class initializes its modules.
        # from_pretrained may leave missing CRF checkpoint weights uninitialized.
        model.crf.reset_parameters()
    for name, parameter in model.named_parameters():
        if not torch.isfinite(parameter).all():
            raise FloatingPointError(f'Non-finite model parameter before training: {name}')
    if cfg['gradient_checkpointing']:
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
    model.to(device)
    if device.type == 'cuda':
        print(f'GPU={torch.cuda.get_device_name(0)}; VRAM={torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GiB; precision={cfg["precision"]}', flush=True)
    else:
        print('Device=CPU; precision=fp32', flush=True)
    return model, tokenizer, device


def optimizer_for(model, cfg):
    encoder = list(model.roberta.named_parameters())
    no_decay = ['bias', 'LayerNorm.weight']
    return torch.optim.AdamW([
        {'params': [p for n, p in encoder if not any(x in n for x in no_decay)], 'weight_decay': cfg['weight_decay']},
        {'params': [p for n, p in encoder if any(x in n for x in no_decay)], 'weight_decay': 0.0},
        {'params': list(model.classifier.parameters()) + list(model.crf.parameters()),
         'lr': cfg['classifier_learning_rate'], 'weight_decay': cfg['weight_decay']}],
        lr=cfg['learning_rate'], eps=cfg['adam_epsilon'])


def train(cfg, smoke=False):
    out = Path(cfg['output_dir'])
    out.mkdir(parents=True, exist_ok=True)
    last = out / 'last_checkpoint.pt'
    if not smoke and (last.exists() or (out / 'best_model.pt').exists()) and not cfg['resume']:
        raise RuntimeError('Run already exists: set resume=True or choose a new run_name.')
    if not smoke and cfg['resume'] and not last.exists():
        raise FileNotFoundError(f'Resume requested but missing {last}')
    model, tokenizer, device = setup(cfg)
    train_set, train_stats = feature_dataset(cfg, 'train', tokenizer)
    dev_set, dev_stats = feature_dataset(cfg, 'dev', tokenizer)
    if smoke:
        # Longest examples exercise near-worst-case padding/VRAM, not just short sentences.
        longest = sorted(range(len(train_set)), key=lambda i: len(train_set[i]['input_ids']), reverse=True)
        train_set = Subset(train_set, longest[:cfg['train_batch_size'] * cfg['smoke_batches']])
        dev_set = Subset(dev_set, range(min(len(dev_set), cfg['eval_batch_size'])))
    iterator = loader(train_set, cfg, tokenizer, train=True)
    steps_per_epoch = math.ceil(len(iterator) / cfg['gradient_accumulation_steps'])
    optimizer = optimizer_for(model, cfg)
    total_steps = steps_per_epoch * cfg['epochs']
    scheduler = get_linear_schedule_with_warmup(optimizer, int(total_steps * cfg['warmup_proportion']), total_steps)
    scaler = torch.amp.GradScaler('cuda', enabled=cfg['precision'] == 'fp16')
    first_epoch, best_score, bad_epochs, history = 0, -1.0, 0, []
    if cfg['resume'] and not smoke:
        # Load only the checkpoint created by this run; it contains a Namespace + RNG state.
        saved = torch.load(last, map_location='cpu', weights_only=False)
        keys = ['model_name_or_path', 'max_seq_length', 'train_batch_size', 'eval_batch_size',
                'gradient_accumulation_steps', 'learning_rate', 'classifier_learning_rate',
                'weight_decay', 'adam_epsilon', 'max_grad_norm', 'warmup_proportion', 'epochs',
                'precision', 'seed', 'selection_metric', 'gradient_checkpointing', 'early_stop',
                'dropout', 'train_limit', 'dev_limit', 'dataset_repo', 'dataset_revision']
        for key in keys:
            if saved['config'][key] != cfg[key]:
                raise ValueError(f'Cannot resume after changing {key}; use a new run_name.')
        if saved['dataset_stats'] != {'train': train_stats, 'dev': dev_stats}:
            raise ValueError('Training/dev data changed; use a new run_name.')
        model.load_state_dict(saved['model'])
        optimizer.load_state_dict(saved['optimizer'])
        scheduler.load_state_dict(saved['scheduler'])
        scaler.load_state_dict(saved['scaler'])
        first_epoch = saved['epoch'] + 1
        best_score, bad_epochs, history = saved['best_score'], saved['bad_epochs'], saved['history']
        random.setstate(saved['rng']['python'])
        np.random.set_state(saved['rng']['numpy'])
        torch.set_rng_state(saved['rng']['torch'])
        if device.type == 'cuda':
            torch.cuda.set_rng_state_all(saved['rng']['cuda'])
        del saved
        print(f'Resume from epoch {first_epoch + 1}; best dev score={best_score:.6f}', flush=True)
    tokenizer.save_pretrained(out / 'tokenizer')
    model.config.save_pretrained(out / 'model_config')
    write_json(out / 'run_config.json', cfg)
    write_json(out / 'dataset_stats.json', {'train': train_stats, 'dev': dev_stats})
    writer = SummaryWriter(str(out / 'tensorboard'))
    for epoch in range(first_epoch, 1 if smoke else cfg['epochs']):
        if bad_epochs >= cfg['early_stop']:
            print('Early stopping reached.', flush=True)
            break
        iterator = loader(train_set, cfg, tokenizer, train=True, seed=cfg['seed'] + epoch)
        model.train()
        optimizer.zero_grad(set_to_none=True)
        total_loss, started = 0.0, time.time()
        if device.type == 'cuda':
            torch.cuda.reset_peak_memory_stats()
        bar = tqdm(iterator, desc=f'Epoch {epoch + 1}/{cfg["epochs"]}')
        for step, batch in enumerate(bar):
            batch = gpu_batch(batch, device)
            # Preserve baseline CRF reduction='sum'. With accumulation, sum microbatch
            # losses to match a larger physical batch, including the last partial batch.
            with amp(cfg, device):
                output = model(**batch)
            if not torch.isfinite(output.loss):
                raise FloatingPointError(
                    f'Non-finite loss at epoch={epoch + 1}, batch={step + 1}, '
                    f'precision={cfg["precision"]}; inspect model parameters and batch data.')
            total_loss += output.loss.item()
            scaler.scale(output.loss).backward()
            if (step + 1) % cfg['gradient_accumulation_steps'] == 0 or step + 1 == len(iterator):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg['max_grad_norm'],
                                               error_if_nonfinite=cfg['precision'] != 'fp16')
                old_scale = scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()
                if scaler.get_scale() >= old_scale:
                    scheduler.step()
                optimizer.zero_grad(set_to_none=True)
            bar.set_postfix(loss=f'{output.loss.item():.2f}', lr=f'{optimizer.param_groups[0]["lr"]:.2e}')
        metrics, _, _ = evaluate(model, loader(dev_set, cfg, tokenizer), cfg, device)
        record = {'epoch': epoch + 1, 'train_loss_per_sentence': total_loss / len(train_set),
                  **metrics, 'seconds': time.time() - started,
                  'peak_vram_gib': torch.cuda.max_memory_allocated() / 2**30 if device.type == 'cuda' else 0.0}
        print(json.dumps(record, ensure_ascii=False), flush=True)
        if smoke:
            write_json(out / 'smoke_metrics.json', record)
            writer.close()
            return
        history.append(record)
        for key, val in record.items():
            if key != 'epoch':
                writer.add_scalar(key, val, epoch + 1)
        writer.flush()
        score = metrics[cfg['selection_metric']]
        if score > best_score:
            best_score, bad_epochs = score, 0
            save_checkpoint(out / 'best_model.pt', model, cfg, epoch=epoch, metrics=metrics)
        else:
            bad_epochs += 1
        write_json(out / 'history.json', history)
        save_checkpoint(last, model, cfg, epoch=epoch, best_score=best_score,
                        bad_epochs=bad_epochs, history=history,
                        dataset_stats={'train': train_stats, 'dev': dev_stats},
                        optimizer=optimizer.state_dict(), scheduler=scheduler.state_dict(),
                        scaler=scaler.state_dict(), rng={'python': random.getstate(),
                        'numpy': np.random.get_state(), 'torch': torch.get_rng_state(),
                        'cuda': torch.cuda.get_rng_state_all() if device.type == 'cuda' else []})
    writer.close()
    print(f'Training finished. Best dev {cfg["selection_metric"]}={best_score:.6f}', flush=True)
''')

## 7. Runner — phần D: test, inference và CLI


In [7]:
RUNNER_PARTS.append(r'''
def test(cfg):
    out = Path(cfg['output_dir'])
    saved = torch.load(out / 'best_model.pt', map_location='cpu', weights_only=False)
    actual_cfg = dict(saved['config'])
    if (actual_cfg['dataset_repo'], actual_cfg['dataset_revision']) != (cfg['dataset_repo'], cfg['dataset_revision']):
        raise ValueError('Test dataset revision differs from the training checkpoint.')
    actual_cfg.update({k: cfg[k] for k in ['data_files', 'cache_dir', 'output_dir', 'eval_batch_size', 'num_workers']})
    model, tokenizer, device = setup(actual_cfg, pretrained=False)
    model.load_state_dict(saved['model'])
    del saved
    dataset, stats = feature_dataset(actual_cfg, 'test', tokenizer)
    metrics, gold, pred = evaluate(model, loader(dataset, actual_cfg, tokenizer), actual_cfg, device)
    write_json(out / 'test_metrics.json', {'dataset_repo': actual_cfg['dataset_repo'],
                                           'dataset_revision': actual_cfg['dataset_revision'],
                                           'metrics': metrics, 'dataset': stats})
    write_json(out / 'test_predictions.json', {'gold': gold, 'pred': pred})
    report = entity_report(gold, pred, mode='strict', scheme=IOB2, digits=4, zero_division=0)
    report += '\nBIO token report\n' + classification_report(
        [x for s in gold for x in s], [x for s in pred for x in s], labels=LABELS, digits=4, zero_division=0)
    (out / 'test_report.txt').write_text(report, encoding='utf-8')
    print(json.dumps(metrics, indent=2), flush=True)
    print(report, flush=True)


def predict(cfg, text):
    # Input must already be word-segmented (underscores), as in PAP_NER.
    out = Path(cfg['output_dir'])
    saved = torch.load(out / 'best_model.pt', map_location='cpu', weights_only=False)
    actual_cfg = saved['config']
    model, tokenizer, device = setup(actual_cfg, pretrained=False)
    model.load_state_dict(saved['model'])
    model.eval()
    words = text.split()
    if not words:
        raise ValueError('Empty input')
    ex = encode_words(words, [0] * len(words), tokenizer, actual_cfg['max_seq_length'])
    if len(ex['labels']) < len(words):
        print(f'Input truncated: keeping {len(ex["labels"])} / {len(words)} words.', flush=True)
    batch = gpu_batch(Collator(tokenizer.pad_token_id)([ex]), device)
    with torch.inference_mode(), amp(actual_cfg, device):
        tags = model(**batch, decode=True).tags[0]
    for word, tag in zip(words, tags):
        print(f'{word:45} {LABELS[tag]}')


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('mode', choices=['train', 'smoke', 'test', 'predict'])
    parser.add_argument('--config', required=True)
    parser.add_argument('--text', default='')
    args = parser.parse_args()
    cfg = json.loads(Path(args.config).read_text(encoding='utf-8'))
    status = Path(cfg['output_dir']) / f'{args.mode}_status.json'
    status.parent.mkdir(parents=True, exist_ok=True)
    write_json(status, {'state': 'running', 'pid': os.getpid()})
    try:
        if args.mode in ['train', 'smoke']:
            train(cfg, smoke=args.mode == 'smoke')
        elif args.mode == 'test':
            test(cfg)
        else:
            predict(cfg, args.text)
        write_json(status, {'state': 'completed', 'pid': os.getpid()})
    except BaseException as exc:
        write_json(status, {'state': 'failed', 'pid': os.getpid(), 'error': str(exc)})
        traceback.print_exc()
        if isinstance(exc, torch.cuda.OutOfMemoryError):
            print('OOM: giảm physical train batch nhưng giữ effective batch = 32 bằng gradient accumulation; '
                  'ví dụ batch=4, accumulation=8. Giảm eval_batch_size nếu cần. Use a new run_name.', flush=True)
        raise


if __name__ == '__main__':
    main()
''')

## 8. Ghép runner và kiểm tra syntax


In [8]:
RUNNER_SOURCE = "".join(RUNNER_PARTS)
RUNNER.write_text(RUNNER_SOURCE, encoding="utf-8")

subprocess.run([str(ENV_PYTHON), "-m", "py_compile", str(RUNNER)], check=True, env=ENV)

freeze = subprocess.check_output(
    [str(ENV_PYTHON), "-m", "pip", "freeze"],
    text=True,
    env=ENV,
)
(RUN_DIR / "requirements-frozen.txt").write_text(freeze, encoding="utf-8")

print("Runner:", RUNNER)
print("Runner syntax: OK")

Runner: /kaggle/working/ner-pap/papner_runner.py
Runner syntax: OK


## 9. Smoke test GPU trước khi train

Smoke test lấy các câu dài nhất để kiểm tra gần worst-case về VRAM trước khi chạy cả epoch.

Notebook tự chọn batch theo VRAM và luôn giữ **effective batch = 32**. Nếu smoke test vẫn OOM trên một GPU Kaggle đặc biệt, giảm `TRAIN_BATCH_SIZE` và tăng `GRAD_ACCUM_STEPS` tương ứng trong cell cấu hình.


In [9]:
smoke_cfg = dict(
    CFG,
    resume=False,
    output_dir=str(WORK_ROOT / "smoke" / RUN_NAME),
)
smoke_path = WORK_ROOT / f"smoke-{RUN_NAME}.json"
smoke_path.write_text(json.dumps(smoke_cfg, ensure_ascii=False, indent=2), encoding="utf-8")

subprocess.run(
    [str(ENV_PYTHON), "-u", str(RUNNER), "smoke", "--config", str(smoke_path)],
    check=True,
    env=ENV,
)

smoke_metrics_path = Path(smoke_cfg["output_dir"]) / "smoke_metrics.json"
print(smoke_metrics_path.read_text(encoding="utf-8"))

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Some weights of PhoBertCrf were not initialized from the model checkpoint at vinai/phobert-base-v2 and are newly initialized: ['classifier.bias', 'classifier.weight', 'crf.end_transitions', 'crf.start_transitions', 'crf.transitions']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


GPU=Tesla T4; VRAM=14.6 GiB; precision=fp16


Tokenize train: 152it [00:00, 764.64it/s]

Tokenize train: 428it [00:00, 898.63it/s]

...

Epoch 1/1:   0%|          | 0/2 [00:00<?, ?it/s]

Epoch 1/1:  50%|█████     | 1/2 [00:00<00:00,  1.50it/s, loss=1744.21, lr=2.00e-05]

Evaluate: 100%|██████████| 1/1 [00:00<00:00, 10.06it/s]


{"epoch": 1, "train_loss_per_sentence": 188.62245178222656, "loss_per_sentence": 49.51879119873047, "bio_accuracy": 0.0220125786163522, "bio_macro_f1": 0.023514735693123345, "entity_strict_precision_micro": 0.0, "entity_strict_recall_micro": 0.0, "entity_strict_micro_f1": 0.0, "entity_strict_macro_f1": 0.0, "seconds": 1.0403807163238525, "peak_vram_gib": 1.897867202758789}


{
  "epoch": 1,
  "train_loss_per_sentence": 188.62245178222656,
  "loss_per_sentence": 49.51879119873047,
  "bio_accuracy": 0.0220125786163522,
  "bio_macro_f1": 0.023514735693123345,
  "entity_strict_precision_micro": 0.0,
  "entity_strict_recall_micro": 0.0,
  "entity_strict_micro_f1": 0.0,
  "entity_strict_macro_f1": 0.0,
  "seconds": 1.0403807163238525,
  "peak_vram_gib": 1.897867202758789
}


## 10. Train 1 epoch — verification run

Run này chỉ train **1 epoch** để xác nhận:

- PhoBERT-v2 tải đúng.
- PAP_NER tải và parse đúng.
- Subword alignment + CRF loss chạy đúng.
- Backprop/optimizer/scheduler chạy được trên GPU Kaggle.
- Dev evaluation và checkpoint được tạo.
- Sau đó có thể chạy test + advanced evaluation.

Khi 1 epoch chạy ổn, chỉ cần đổi `TRIAL_EPOCHS` ở cell cấu hình để chạy benchmark dài hơn.


In [10]:
TRAIN_LOG = RUN_DIR / "train.log"
PID_PATH = RUN_DIR / "train.pid"
STATUS_PATH = RUN_DIR / "train_status.json"

def process_alive(pid):
    try:
        os.kill(pid, 0)
        stat = Path(f"/proc/{pid}/stat")
        return not (stat.exists() and stat.read_text().split(") ", 1)[1].startswith("Z"))
    except (ProcessLookupError, PermissionError, FileNotFoundError):
        return False

pid = int(PID_PATH.read_text()) if PID_PATH.exists() else None
cmdline = Path(f"/proc/{pid}/cmdline") if pid else None
is_our_process = bool(
    pid
    and process_alive(pid)
    and cmdline.exists()
    and str(CONFIG_PATH).encode() in cmdline.read_bytes()
)

if is_our_process:
    print(f"Đang nối vào process train PID={pid}")
else:
    last = RUN_DIR / "last_checkpoint.pt"
    best = RUN_DIR / "best_model.pt"

    if CFG["resume"]:
        assert last.exists(), f"Không có checkpoint để resume: {last}"
    elif last.exists() or best.exists():
        raise RuntimeError(
            "Run đã có checkpoint. Đặt RESUME=True hoặc đổi RUN_NAME rồi chạy lại từ cell cấu hình."
        )

    CONFIG_PATH.write_text(json.dumps(CFG, ensure_ascii=False, indent=2), encoding="utf-8")
    STATUS_PATH.unlink(missing_ok=True)

    with TRAIN_LOG.open("ab", buffering=0) as log:
        train_process = subprocess.Popen(
            [str(ENV_PYTHON), "-u", str(RUNNER), "train", "--config", str(CONFIG_PATH)],
            stdout=log,
            stderr=subprocess.STDOUT,
            stdin=subprocess.DEVNULL,
            env=ENV,
            cwd=str(WORK_ROOT),
            start_new_session=True,
        )

    pid = train_process.pid
    PID_PATH.write_text(str(pid), encoding="utf-8")
    print(f"Train PID={pid}")
    print(f"Log: {TRAIN_LOG}")

Train PID=162
Log: /kaggle/working/ner-pap/runs/papner_phobert_base_v2_crf_1ep_seed42_kaggle/train.log


## 11. Theo dõi training đến khi hoàn tất


In [11]:
from IPython.display import clear_output
import time

def log_tail(path, limit=12000):
    if not path.exists():
        return "Đang chờ log..."
    with path.open("rb") as stream:
        stream.seek(max(0, path.stat().st_size - limit))
        return stream.read().decode("utf-8", errors="replace").replace("\r", "\n")

while True:
    clear_output(wait=True)
    print(log_tail(TRAIN_LOG))

    status = json.loads(STATUS_PATH.read_text()) if STATUS_PATH.exists() else {}

    if status.get("state") == "completed":
        print("\nTrain hoàn tất.")
        print("Best model:", RUN_DIR / "best_model.pt")
        break

    if status.get("state") == "failed":
        raise RuntimeError(f"Train lỗi: {status.get('error')}. Xem train.log.")

    if not process_alive(pid):
        raise RuntimeError("Process train đã dừng trước khi hoàn tất. Xem train.log.")

    time.sleep(15)

8/645 [00:06<00:18, 25.04it/s]
Evaluate: 100%|██████████| 645/645 [00:24<00:00, 25.89it/s]
{"epoch": 1, "train_loss_per_sentence": 1.3902715053332064, "loss_per_sentence": 2.563051732196095, "bio_accuracy": 0.99042986176467, "bio_macro_f1": 0.9725020842397022, "entity_strict_precision_micro": 0.9859607091518927, "entity_strict_recall_micro": 0.9623065051676566, "entity_strict_micro_f1": 0.9739900125434882, "entity_strict_macro_f1": 0.9781262336517285, "seconds": 1667.9758653640747, "peak_vram_gib": 2.8993420600891113}
Training finished. Best dev entity_strict_micro_f1=0.973990


Train hoàn tất.
Best model: /kaggle/working/ner-pap/runs/papner_phobert_base_v2_crf_1ep_seed42_kaggle/best_model.pt


## 12. Lịch sử train và đồ thị


### Post-training summary (text only)
Bản này bỏ hoàn toàn bước vẽ biểu đồ. Sau train notebook chỉ in metrics dạng text rồi tiếp tục test/evaluation.


In [12]:
history_path = RUN_DIR / "history.json"
history = json.loads(history_path.read_text(encoding="utf-8"))

print(f"Notebook build: {NOTEBOOK_BUILD}")
print("=== TRAINING SUMMARY ===")

for row in history:
    print(
        f"Epoch {row['epoch']:2d}: "
        f"Strict Micro F1={row['entity_strict_micro_f1']:.4%} | "
        f"Strict Macro F1={row['entity_strict_macro_f1']:.4%} | "
        f"BIO Macro F1={row['bio_macro_f1']:.4%} | "
        f"VRAM={row['peak_vram_gib']:.2f} GiB | "
        f"time={row['seconds'] / 60:.1f} min"
    )

print("Plotting: DISABLED")
print("Continuing to test/evaluation...")


Notebook build: FINAL_NO_PLOT_V4
=== TRAINING SUMMARY ===
Epoch  1: Strict Micro F1=97.3990% | Strict Macro F1=97.8126% | BIO Macro F1=97.2502% | VRAM=2.90 GiB | time=27.8 min
Plotting: DISABLED
Continuing to test/evaluation...


## 13. Đánh giá trên test split

Best checkpoint được đánh giá một lần trên test sau khi training hoàn tất.

Output chính:

- `test_metrics.json`
- `test_predictions.json`
- `test_report.txt`

`test_report.txt` có strict entity report theo từng loại entity và BIO token report.


In [13]:
assert (RUN_DIR / "best_model.pt").exists(), "Chưa có best_model.pt"

subprocess.run(
    [str(ENV_PYTHON), "-u", str(RUNNER), "test", "--config", str(CONFIG_PATH)],
    check=True,
    env=ENV,
)

print((RUN_DIR / "test_metrics.json").read_text(encoding="utf-8"))

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


GPU=Tesla T4; VRAM=14.6 GiB; precision=fp16


Tokenize test: 76it [00:00, 750.41it/s]

Tokenize test: 276it [00:00, 945.49it/s]

Tokenize test: 499it [00:00, 1045.88it/s]

...

Evaluate:  99%|█████████▊| 634/643 [00:25<00:00, 24.22it/s]

Evaluate: 100%|█████████▉| 640/643 [00:25<00:00, 25.32it/s]

Evaluate: 100%|██████████| 643/643 [00:26<00:00, 24.71it/s]


{
  "loss_per_sentence": 2.263437088105576,
  "bio_accuracy": 0.9912895704715619,
  "bio_macro_f1": 0.9734481279099942,
  "entity_strict_precision_micro": 0.9848764831744796,
  "entity_strict_recall_micro": 0.965486008485484,
  "entity_strict_micro_f1": 0.9750848559254712,
  "entity_strict_macro_f1": 0.974484670535464
}
              precision    recall  f1-score   support

          CQ     0.9880    0.9676    0.9777      7311
          NG     0.9998    0.9998    0.9998      4021
          SL     0.9740    0.9597    0.9668      1638
        VBPL     0.9618    0.9722    0.9670       828
          ĐT     0.9784    0.9447    0.9612      7179

   micro avg     0.9849    0.9655    0.9751     20977
   macro avg     0.9804    0.9688    0.9745     20977
weighted avg     0.9848    0.9655    0.9750     20977

BIO token report
              precision    recall  f1-score   support

           O     0.9919    0.9981    0.9950    205461
        B-CQ     0.9908    0.9703    0.9804      7311
        I

{
  "dataset_repo": "Leekien0108/PAP_NER",
  "dataset_revision": "7a0f6b1233667702925baa69c07c55d5d26eb6cd",
  "metrics": {
    "loss_per_sentence": 2.263437088105576,
    "bio_accuracy": 0.9912895704715619,
    "bio_macro_f1": 0.9734481279099942,
    "entity_strict_precision_micro": 0.9848764831744796,
    "entity_strict_recall_micro": 0.965486008485484,
    "entity_strict_micro_f1": 0.9750848559254712,
    "entity_strict_macro_f1": 0.974484670535464
  },
  "dataset": {
    "truncated_sentences": 0,
    "discarded_words": 0,
    "sha256": "1fe8b2992b00bec2457d96c20831421e7a6b9fe5b3b62320f9d0efd1217bde2a",
    "limit": null
  }
}


## 14. Đánh giá nâng cao theo kiểu paper (Strict / Partial / Entity Type / Exact)

Cell này đọc `test_predictions.json` và dựng bảng tương tự hình bạn gửi: theo từng tag và có cả dòng **Micro / Macro** cho 4 chế độ đánh giá.


In [14]:

from IPython.display import HTML, display

assert (RUN_DIR / "test_predictions.json").exists(), "Chưa có test_predictions.json. Hãy chạy cell đánh giá test trước."

advanced_eval_script = r"""
import json
from pathlib import Path

import pandas as pd
from nervaluate import Evaluator

run_dir = Path(r"__RUN_DIR__")
pred_path = run_dir / "test_predictions.json"
raw = json.loads(pred_path.read_text(encoding="utf-8"))
true_seqs = raw["gold"]
pred_seqs = raw["pred"]
entity_tags = ["CQ", "ĐT", "VBPL", "NG", "SL"]

evaluator = Evaluator(true_seqs, pred_seqs, tags=entity_tags, loader="list")
results, results_by_tag, result_indices, result_indices_by_tag = evaluator.evaluate()

modes = [
    ("strict", "Strict"),
    ("partial", "Partial"),
    ("ent_type", "Entity Type"),
    ("exact", "Exact"),
]


def unpack_metrics(obj):
    if isinstance(obj, dict):
        src = obj
    else:
        src = {
            name: getattr(obj, name)
            for name in [
                "correct", "incorrect", "partial", "missed", "spurious",
                "possible", "actual", "precision", "recall", "f1"
            ]
            if hasattr(obj, name)
        }
    out = {}
    for key, value in src.items():
        if isinstance(value, (int, float)):
            out[key] = float(value)
        else:
            out[key] = value
    return out

rows = []
for tag in entity_tags:
    row = {("Tag", ""): tag}
    tag_block = results_by_tag.get(tag) or {mode_key: {} for mode_key, _ in modes}
    for mode_key, mode_name in modes:
        metrics = unpack_metrics(tag_block[mode_key])
        row[(mode_name, "P")] = metrics.get("precision", 0.0) * 100.0
        row[(mode_name, "R")] = metrics.get("recall", 0.0) * 100.0
        row[(mode_name, "F1")] = metrics.get("f1", 0.0) * 100.0
    rows.append(row)

micro_row = {("Tag", ""): "Micro"}
for mode_key, mode_name in modes:
    metrics = unpack_metrics(results[mode_key])
    micro_row[(mode_name, "P")] = metrics.get("precision", 0.0) * 100.0
    micro_row[(mode_name, "R")] = metrics.get("recall", 0.0) * 100.0
    micro_row[(mode_name, "F1")] = metrics.get("f1", 0.0) * 100.0
rows.append(micro_row)

macro_row = {("Tag", ""): "Macro"}
for mode_key, mode_name in modes:
    p_vals = [r[(mode_name, "P")] for r in rows[:-1]]
    r_vals = [r[(mode_name, "R")] for r in rows[:-1]]
    f_vals = [r[(mode_name, "F1")] for r in rows[:-1]]
    macro_row[(mode_name, "P")] = sum(p_vals) / len(p_vals)
    macro_row[(mode_name, "R")] = sum(r_vals) / len(r_vals)
    macro_row[(mode_name, "F1")] = sum(f_vals) / len(f_vals)
rows.append(macro_row)

columns = [("Tag", "")]
for _, mode_name in modes:
    columns.extend([(mode_name, "P"), (mode_name, "R"), (mode_name, "F1")])

df = pd.DataFrame(rows)[columns]
flat_df = df.copy()
flat_df.columns = [
    "Tag" if a == "Tag" else f"{a}_{b}"
    for a, b in flat_df.columns
]

csv_path = run_dir / "advanced_eval_table.csv"
html_path = run_dir / "advanced_eval_table.html"
json_path = run_dir / "advanced_eval_summary.json"
md_path = run_dir / "advanced_eval_table.md"

flat_df.to_csv(csv_path, index=False, encoding="utf-8-sig")

styled = (
    df.style
    .format(precision=2, na_rep="-")
    .hide(axis="index")
    .set_caption("PAP_NER evaluation table — Strict / Partial / Entity Type / Exact")
)
html_path.write_text(styled.to_html(), encoding="utf-8")
md_path.write_text(flat_df.to_markdown(index=False, floatfmt=".2f"), encoding="utf-8")
json_path.write_text(json.dumps({
    "overall": {k: unpack_metrics(v) for k, v in results.items()},
    "by_tag": {
        tag: {mode_key: unpack_metrics(mode_block[mode_key]) for mode_key, _ in modes}
        for tag, mode_block in results_by_tag.items()
    },
}, ensure_ascii=False, indent=2), encoding="utf-8")

print(csv_path)
print(html_path)
print(json_path)
print(md_path)
""".replace("__RUN_DIR__", str(RUN_DIR))

subprocess.run([str(ENV_PYTHON), "-c", advanced_eval_script], check=True, env=ENV)

for path in [
    RUN_DIR / "advanced_eval_table.csv",
    RUN_DIR / "advanced_eval_table.html",
    RUN_DIR / "advanced_eval_summary.json",
    RUN_DIR / "advanced_eval_table.md",
]:
    assert path.exists(), f"Thiếu file: {path}"

print("Đã tạo các file đánh giá nâng cao:")
print("-", RUN_DIR / "advanced_eval_table.csv")
print("-", RUN_DIR / "advanced_eval_table.html")
print("-", RUN_DIR / "advanced_eval_summary.json")
print("-", RUN_DIR / "advanced_eval_table.md")

display(HTML((RUN_DIR / "advanced_eval_table.html").read_text(encoding="utf-8")))
print((RUN_DIR / "advanced_eval_table.md").read_text(encoding="utf-8"))


/kaggle/working/ner-pap/runs/papner_phobert_base_v2_crf_1ep_seed42_kaggle/advanced_eval_table.csv
/kaggle/working/ner-pap/runs/papner_phobert_base_v2_crf_1ep_seed42_kaggle/advanced_eval_table.html
/kaggle/working/ner-pap/runs/papner_phobert_base_v2_crf_1ep_seed42_kaggle/advanced_eval_summary.json
/kaggle/working/ner-pap/runs/papner_phobert_base_v2_crf_1ep_seed42_kaggle/advanced_eval_table.md
Đã tạo các file đánh giá nâng cao:
- /kaggle/working/ner-pap/runs/papner_phobert_base_v2_crf_1ep_seed42_kaggle/advanced_eval_table.csv
- /kaggle/working/ner-pap/runs/papner_phobert_base_v2_crf_1ep_seed42_kaggle/advanced_eval_table.html
- /kaggle/working/ner-pap/runs/papner_phobert_base_v2_crf_1ep_seed42_kaggle/advanced_eval_summary.json
- /kaggle/working/ner-pap/runs/papner_phobert_base_v2_crf_1ep_seed42_kaggle/advanced_eval_table.md


"('Tag', '')","('Strict', 'P')","('Strict', 'R')","('Strict', 'F1')","('Partial', 'P')","('Partial', 'R')","('Partial', 'F1')","('Entity Type', 'P')","('Entity Type', 'R')","('Entity Type', 'F1')","('Exact', 'P')","('Exact', 'R')","('Exact', 'F1')"
CQ,98.74,96.77,97.75,98.97,97.00,97.98,99.20,97.22,98.20,98.74,96.77,97.75
ĐT,97.29,94.47,95.86,98.39,95.54,96.94,99.45,96.57,97.99,97.29,94.47,95.86
VBPL,95.51,97.71,96.60,96.64,98.85,97.73,97.76,100.00,98.87,95.51,97.71,96.60
NG,99.93,99.98,99.95,99.93,99.98,99.95,99.93,99.98,99.95,99.93,99.98,99.95
SL,96.21,95.85,96.03,97.80,97.44,97.62,99.39,99.02,99.21,96.21,95.85,96.03
Micro,98.15,96.56,97.35,98.77,97.18,97.97,99.38,97.78,98.58,98.15,96.56,97.35
Macro,97.54,96.96,97.24,98.34,97.76,98.04,99.15,98.56,98.84,97.54,96.96,97.24


| Tag   |   Strict_P |   Strict_R |   Strict_F1 |   Partial_P |   Partial_R |   Partial_F1 |   Entity Type_P |   Entity Type_R |   Entity Type_F1 |   Exact_P |   Exact_R |   Exact_F1 |
|:------|-----------:|-----------:|------------:|------------:|------------:|-------------:|----------------:|----------------:|-----------------:|----------:|----------:|-----------:|
| CQ    |      98.74 |      96.77 |       97.75 |       98.97 |       97.00 |        97.98 |           99.20 |           97.22 |            98.20 |     98.74 |     96.77 |      97.75 |
| ĐT    |      97.29 |      94.47 |       95.86 |       98.39 |       95.54 |        96.94 |           99.45 |           96.57 |            97.99 |     97.29 |     94.47 |      95.86 |
| VBPL  |      95.51 |      97.71 |       96.60 |       96.64 |       98.85 |        97.73 |           97.76 |          100.00 |            98.87 |     95.51 |     97.71 |      96.60 |
| NG    |      99.93 |      99.98 |       99.95 |       99.93 |       99.98

## 15. Inference thử với một câu đã word-segment

PAP_NER đã được word-segment. Với văn bản thô mới, hãy chạy VnCoreNLP/RDRSegmenter trước khi đưa vào cell này.


In [15]:
SEGMENTED_TEXT = (
    "Uỷ_ban_nhân_dân tỉnh Thái_Bình ban_hành Nghị_định 123/2020/NĐ-CP "
    "ngày 15 tháng 10 năm 2020 về đăng_ký doanh_nghiệp với lệ_phí 5 triệu đồng ."
)

subprocess.run(
    [
        str(ENV_PYTHON),
        "-u",
        str(RUNNER),
        "predict",
        "--config",
        str(CONFIG_PATH),
        "--text",
        SEGMENTED_TEXT,
    ],
    check=True,
    env=ENV,
)

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


GPU=Tesla T4; VRAM=14.6 GiB; precision=fp16


Uỷ_ban_nhân_dân                               B-CQ
tỉnh                                          I-CQ
Thái_Bình                                     O
ban_hành                                      O
Nghị_định                                     B-VBPL
123/2020/NĐ-CP                                I-VBPL
ngày                                          O
15                                            O
tháng                                         O
10                                            O
năm                                           O
2020                                          O
về                                            O
đăng_ký                                       O
doanh_nghiệp                                  B-ĐT
với                                           O
lệ_phí                                        O
5                                             B-SL
triệu                                         I-SL
đồng                                          I-SL
.           

CompletedProcess(args=['/usr/bin/python', '-u', '/kaggle/working/ner-pap/papner_runner.py', 'predict', '--config', '/kaggle/working/ner-pap/runs/papner_phobert_base_v2_crf_1ep_seed42_kaggle/launch_config.json', '--text', 'Uỷ_ban_nhân_dân tỉnh Thái_Bình ban_hành Nghị_định 123/2020/NĐ-CP ngày 15 tháng 10 năm 2020 về đăng_ký doanh_nghiệp với lệ_phí 5 triệu đồng .'], returncode=0)

## 16. Đóng gói kết quả


In [16]:
import zipfile
from IPython.display import FileLink, display

ARCHIVE_PATH = NOTEBOOK_DIR / f"{RUN_NAME}-results.zip"

items = [
    "best_model.pt",
    "run_config.json",
    "launch_config.json",
    "dataset_stats.json",
    "history.json",
    "test_metrics.json",
    "test_predictions.json",
    "test_report.txt",
    "advanced_eval_table.csv",
    "advanced_eval_table.html",
    "advanced_eval_summary.json",
    "advanced_eval_table.md",
    "requirements-frozen.txt",
]

with zipfile.ZipFile(
    ARCHIVE_PATH,
    "w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=1,
) as archive:
    for name in items:
        path = RUN_DIR / name
        if path.exists():
            archive.write(path, arcname=f"{RUN_NAME}/{name}")

    for directory in ["tokenizer", "model_config"]:
        base = RUN_DIR / directory
        if base.exists():
            for path in base.rglob("*"):
                if path.is_file():
                    archive.write(
                        path,
                        arcname=f"{RUN_NAME}/{path.relative_to(RUN_DIR)}",
                    )

    archive.write(RUNNER, arcname=f"{RUN_NAME}/papner_runner.py")

print(f"ZIP: {ARCHIVE_PATH} ({ARCHIVE_PATH.stat().st_size / 2**20:.1f} MiB)")
display(FileLink(str(ARCHIVE_PATH)))
print("Resume checkpoint:", RUN_DIR / "last_checkpoint.pt")


ZIP: /kaggle/working/papner_phobert_base_v2_crf_1ep_seed42_kaggle-results.zip (479.0 MiB)


/kaggle/working/papner_phobert_base_v2_crf_1ep_seed42_kaggle-results.zip

Resume checkpoint: /kaggle/working/ner-pap/runs/papner_phobert_base_v2_crf_1ep_seed42_kaggle/last_checkpoint.pt


## 17. Tùy chọn: dừng process train

Cell này mặc định **không làm gì**. Chỉ đổi `STOP_TRAIN=True` khi thật sự muốn dừng.


In [17]:
STOP_TRAIN = False

if STOP_TRAIN:
    import signal

    current_pid = int(PID_PATH.read_text())
    current_cmd = Path(f"/proc/{current_pid}/cmdline")

    if (
        process_alive(current_pid)
        and current_cmd.exists()
        and str(CONFIG_PATH).encode() in current_cmd.read_bytes()
    ):
        os.kill(current_pid, signal.SIGTERM)
        print(f"Đã gửi SIGTERM tới PID={current_pid}")
    else:
        print("Không có process train của run này đang chạy.")